<a href="https://colab.research.google.com/github/alicex2020/Mandarin-Tone-Classification/blob/master/Mandarin_Tone_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CODE BASED OF https://github.com/alicex2020/Mandarin-Tone-Classification

In [ ]:
# import the most useful packages
import numpy as np 
import matplotlib
import math
import os
from matplotlib import pyplot as plt
import IPython.display as ipd
print('finished importing')
import librosa
import librosa.display

In [ ]:
# # make a folder for this course and this project. 
# # the -p flag says make any needed parent folders and  don't complain about existing ones.
# my_dir = '/content/drive/My Drive/IW06/AliceProject'
# ! mkdir -p '$my_dir'
# # this command is equivalent to "cd" that sets a working directory
# os.chdir(my_dir)
# !ls '/content/drive/My Drive/IW06/AliceProject/Data'

os.chdir("C:\\Users\\dante\\Chinese_Tone_Detection")
print(os.getcwd())

# **Different Sound Representations for Sample Files**

In [ ]:
# Get audio
file1 = "Classification/training_data/tone_perfect_samples/ma1_FV1_MP3.mp3"
audio1, sample_rate1 = librosa.core.load(file1)
file2 = "Classification/training_data/tone_perfect_samples/ma2_FV1_MP3.mp3"
audio2, sample_rate2 = librosa.core.load(file2)
file3 = "Classification/training_data/tone_perfect_samples/ma3_FV1_MP3.mp3"
audio3, sample_rate3 = librosa.core.load(file3)
file4 = "Classification/training_data/tone_perfect_samples/ma4_FV1_MP3.mp3"
audio4, sample_rate4 = librosa.core.load(file4)



In [ ]:
# spectrograms
X = librosa.stft(audio1)
Xdb = librosa.amplitude_to_db(abs(X))
plt.figure(figsize=(7, 7))
librosa.display.specshow(Xdb)

X2 = librosa.stft(audio2)
Xdb2 = librosa.amplitude_to_db(abs(X2))
plt.figure(figsize=(7,7))
librosa.display.specshow(Xdb2)

X3 = librosa.stft(audio3)
Xdb3 = librosa.amplitude_to_db(abs(X3))
plt.figure(figsize=(7,7))
librosa.display.specshow(Xdb3)

X4 = librosa.stft(audio4)
Xdb4 = librosa.amplitude_to_db(abs(X4))
plt.figure(figsize=(7,7))
print(Xdb4.shape)
# librosa.display.specshow(Xdb4)

In [ ]:
# mel-spectrograms
import matplotlib.cm as cm

mel1 = librosa.feature.melspectrogram(y =audio1, sr=sample_rate1, n_fft=1024, hop_length=512, n_mels=80, fmin=75, fmax=3700)
plt.imshow(np.log10(mel1 + 1e-10), aspect='auto', cmap=cm.plasma)
plt.show()
print(mel1.shape)

mel2 = librosa.feature.melspectrogram(y =audio2, sr=sample_rate2, n_fft=1024, hop_length=512, n_mels=80, fmin=75, fmax=3700)
plt.imshow(np.log10(mel2 + 1e-10), aspect='auto', cmap=cm.plasma)
plt.show()
print(mel2.shape)

mel3 = librosa.feature.melspectrogram(y =audio3, sr=sample_rate3, n_fft=1024, hop_length=512, n_mels=80, fmin=75, fmax=3700)
plt.imshow(np.log10(mel3 + 1e-10), aspect='auto', cmap=cm.plasma)
plt.show()
print(mel3.shape)

mel4 = librosa.feature.melspectrogram(y =audio4, sr=sample_rate4, n_fft=1024, hop_length=512, n_mels=80, fmin=75, fmax=3700)
plt.imshow(np.log10(mel4 + 1e-10), aspect='auto', cmap=cm.plasma)
plt.show()
print(mel4.shape)

In [ ]:
# MFCC

mfcc = librosa.feature.mfcc(y=audio1, sr=sample_rate1, n_mfcc=60)
plt.imshow(mfcc, aspect='auto', cmap=cm.viridis)
plt.show()

mfcc = librosa.feature.mfcc(y=audio2, sr=sample_rate2, n_mfcc=60)
plt.imshow(mfcc, aspect='auto', cmap=cm.viridis)
plt.show()

mfcc = librosa.feature.mfcc(y=audio3, sr=sample_rate3, n_mfcc=60)
plt.imshow(mfcc, aspect='auto', cmap=cm.viridis)
plt.show()

mfcc = librosa.feature.mfcc(y=audio4, sr=sample_rate4, n_mfcc=60)
plt.imshow(mfcc, aspect='auto', cmap=cm.viridis)
plt.show()
print(mfcc.shape)

def mp3tomfcc(file_path, max_pad):
  audio, sample_rate = librosa.core.load(file_path)
  mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=60)
  pad_width = max_pad - mfcc.shape[1]
  mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
  return mfcc
  

In [ ]:
# Pitch

pitch, mag = librosa.core.piptrack(y =audio1, sr=sample_rate1, n_fft=512)
plt.imshow(pitch, aspect='auto')
plt.ylim([20,100])
plt.show()

pitch, mag = librosa.core.piptrack(y =audio2[0:sample_rate2*2], sr=sample_rate2, n_fft=512)
plt.imshow(pitch, aspect='auto')
plt.ylim([20,100])
plt.show()

pitch, mag = librosa.core.piptrack(y =audio3[0:sample_rate3*2], sr=sample_rate3, n_fft=512)
plt.imshow(pitch, aspect='auto')
plt.ylim([20,100])
plt.show()

pitch, mag = librosa.core.piptrack(y =audio4[0:sample_rate4*2], sr=sample_rate4, n_fft=512)
plt.imshow(pitch, aspect='auto')
plt.ylim([20,100])
plt.show()

In [ ]:
#f0
audio_list = [audio1, audio2, audio3, audio4]
sample_list = [sample_rate1, sample_rate2, sample_rate3, sample_rate4]
for i, audio in enumerate(audio_list):
    y = audio
    sr = sample_list[i]

    # Compute F0
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y,
        fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C7')
    )

    # Handle NaNs (important!)
    f0_clean = np.nan_to_num(f0)

    # Time axis
    times = librosa.times_like(f0, sr=sr)

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(times, f0_clean)
    plt.xlabel("Time (s)")
    plt.ylabel("F0 (Hz)")
    plt.title("Pitch Contour (F0)")
    plt.show()

In [ ]:
#f0 preprocessing include f0 and delta f0
import librosa
import numpy as np
import pandas as pd

def mp3tof0(file_path, max_len):
    y, sr = librosa.load(file_path, sr=16000)

    f0, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C7')
    )

    f0 = pd.Series(f0).interpolate().fillna(0).values
    f0 = np.log(f0 + 1e-6)
    f0 = (f0 - np.mean(f0)) / (np.std(f0) + 1e-6)

    # ΔF0
    delta_f0 = np.diff(f0, prepend=f0[0])

    # stack → (T, 2)
    f0 = np.stack([f0, delta_f0], axis=-1)

    # pad
    if f0.shape[0] < max_len:
        pad_width = max_len - f0.shape[0]
        f0 = np.pad(f0, ((0, pad_width), (0, 0)))
    else:
        f0 = f0[:max_len]

    return f0

# Deep Learning on Full Tone Perfect Dataset

In [ ]:
# Compile MFCCs and extract labels: https://github.com/adhishthite/sound-mnist/blob/master/utils/wav2mfcc.py

mfccs = []
file_path = 'Classification/training_data/tone_perfect_samples'


for f in os.listdir(file_path):
  if f.endswith('.mp3'):
    mfccs.append(mp3tomfcc(file_path + '/' + f, 60)) 

  



mfccs = np.asarray(mfccs)
print(mfccs.shape)



In [ ]:
# Save all MFCCs so that don't have to recreate MFCC array every run
np.save('mfccs_all_male_female.npy', mfccs)
mfccs_loaded = np.load('mfccs_all_male_female.npy')
print(mfccs_loaded[0])


In [ ]:
#compile f0s
import sys


f0s = []
file_path = 'Classification/training_data/tone_perfect_samples'

total = len(os.listdir(file_path))
for i, f in enumerate(os.listdir(file_path)):
    if f.endswith('.mp3'):
        f0s.append(mp3tof0(file_path + '/' + f, 60))
        sys.stdout.write(f"\rProcessing file {i}/{total}")
        sys.stdout.flush()

f0s = np.asarray(f0s)

print(f0s.shape)

np.save('f0s_all_male_female.npy', f0s)

f0s_loaded = np.load('f0s_all_male_female.npy')
print(f0s_loaded.shape)
print(f0s_loaded[0])


In [ ]:
# Gather all labels for male and female speakers (1-4 for each of the four tones)
file_path = 'Classification/training_data/tone_perfect_samples'

labels = []

for f in os.listdir(file_path):
  if f.endswith('.mp3'):    
    label = f.split('_')[0][-1] # label is the last character before the first '_'
    labels.append(label)



In [ ]:
from keras.utils import to_categorical
labels = to_categorical(labels, num_classes=None)
print(labels.shape)

In [ ]:
# load labels into drive to avoid having to recreate labels every run
np.save('all_male_female_labels.npy', labels)
labels_loaded = np.load('all_male_female_labels.npy')

In [ ]:
## Optimized Two-Branch CNN: MFCC + F0 with Class Weights ##

import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Conv2D, Conv1D, MaxPooling2D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, Concatenate
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

print("Finished imports")

# --- Load precomputed features ---
mfccs = np.load('Classification/outputs/mfccs_all_male_female.npy')
f0s = np.load('Classification/outputs/f0s_all_male_female.npy')
labels = np.load('Classification/outputs/all_male_female_labels.npy')

num_samples = mfccs.shape[0]
dim_1 = mfccs.shape[1]  # time frames
dim_2 = mfccs.shape[2]  # MFCC coefficients
num_classes = 5

# --- Fix labels ---
if labels.ndim > 1 and labels.shape[1] > 1:
    labels = np.argmax(labels, axis=1)

# One-hot encode labels
y = to_categorical(labels, num_classes=num_classes)
print("Corrected y shape:", y.shape)  # (num_samples, num_classes)

# --- Prepare branch inputs ---
# Normalize MFCCs per sample
X_mfcc = (mfccs - np.mean(mfccs, axis=(1,2), keepdims=True)) / (np.std(mfccs, axis=(1,2), keepdims=True) + 1e-6)
X_mfcc = X_mfcc.reshape((num_samples, dim_1, dim_2, 1))  # 2D CNN
X_f0 = f0s  # 1D CNN over time
print("X_mfcc shape:", X_mfcc.shape)
print("X_f0 shape:", X_f0.shape)

# --- Train/test split ---
X_mfcc_train, X_mfcc_test, X_f0_train, X_f0_test, y_train, y_test = train_test_split(
    X_mfcc, X_f0, y, test_size=0.2, random_state=42
)

# --- Compute class weights ---
y_ints = np.argmax(y_train, axis=1)  # convert one-hot to class indices
class_weights_values = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_ints),
    y=y_ints
)
class_weights = dict(enumerate(class_weights_values))
print("Class weights:", class_weights)

# --- MFCC branch (2D CNN) ---
input_mfcc = Input(shape=(dim_1, dim_2, 1))
x_mfcc = Conv2D(32, (2,2), activation='relu', padding='same')(input_mfcc)
x_mfcc = BatchNormalization()(x_mfcc)
x_mfcc = Conv2D(48, (2,2), activation='relu', padding='same')(x_mfcc)
x_mfcc = BatchNormalization()(x_mfcc)
x_mfcc = MaxPooling2D((2,2))(x_mfcc)
x_mfcc = Dropout(0.25)(x_mfcc)
x_mfcc = Flatten()(x_mfcc)

# --- F0 branch (1D CNN, strengthened) ---
input_f0 = Input(shape=(dim_1, 2))
x_f0 = Conv1D(32, 3, activation='relu', padding='same')(input_f0)
x_f0 = BatchNormalization()(x_f0)
x_f0 = Conv1D(32, 3, activation='relu', padding='same')(x_f0)
x_f0 = BatchNormalization()(x_f0)
x_f0 = MaxPooling1D(2)(x_f0)
x_f0 = Dropout(0.15)(x_f0)
x_f0 = Flatten()(x_f0)

# --- Merge branches ---
merged = Concatenate()([x_mfcc, x_f0])
merged = BatchNormalization()(merged)  # normalize merged features
x = Dense(128, activation='relu')(merged)
x = Dropout(0.25)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(num_classes, activation='softmax')(x)

# --- Create model ---
model = Model(inputs=[input_mfcc, input_f0], outputs=output)
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# --- Train model with class weights ---
history = model.fit(
    [X_mfcc_train, X_f0_train],
    y_train,
    batch_size=32,
    epochs=15,
    validation_data=([X_mfcc_test, X_f0_test], y_test),
    class_weight=class_weights,
    verbose=1
)

# --- Evaluate model ---
loss, acc = model.evaluate([X_mfcc_test, X_f0_test], y_test)
print(f"Test Accuracy: {acc*100:.2f}%")

In [ ]:
model.save('cnn_model_f0_mfcc.h5')
model.save('cnn_model_f0_mfcc.keras')

# Evaluation of Model Performance

In [ ]:
# evaluate model
loss, acc = model.evaluate([X_mfcc_test, X_f0_test], y_test, batch_size=3, verbose=1)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc*100:.2f}%")

In [ ]:
# evaluate model on training
loss, acc = model.evaluate([X_mfcc_train, X_f0_train], y_train, batch_size=3, verbose=1)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc*100:.2f}%")


In [ ]:
import numpy as np
import sklearn.metrics as metrics

# --- Predict using both branches ---
y_pred_ohe = model.predict([X_mfcc_test, X_f0_test])  # pass list of inputs
y_pred_labels = np.argmax(y_pred_ohe, axis=1)         # convert one-hot to class labels

# True labels
y_true_labels = np.argmax(y_test, axis=1)

# Confusion matrix
confusion_matrix = metrics.confusion_matrix(y_true=y_true_labels, y_pred=y_pred_labels)
print("Confusion Matrix:")
print(confusion_matrix)

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

# --- Predict using the two-branch model ---
y_pred_ohe = model.predict([X_mfcc_test, X_f0_test])
y_pred_labels = np.argmax(y_pred_ohe, axis=1)

# True labels
y_true_labels = np.argmax(y_test, axis=1)

# --- Function to plot confusion matrix ---
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    Prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

# --- Compute confusion matrix ---
cm = confusion_matrix(y_true_labels, y_pred_labels)
np.set_printoptions(precision=2)

# --- Plot ---
plt.figure(figsize=(6,6))
plot_confusion_matrix(cm, classes=np.arange(num_classes),
                      title='Confusion Matrix for Two-Branch Model')
plt.show()

In [ ]:
# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()